# 18sA — Canonical source adapters and schema preflight

This first stage isolates all legacy-schema handling. It resolves
the verified March–May and June target, market, deterministic
weather and exact-support files into three canonical source panels.

It specifically supports the verified 18l field
`decision_price_timestamp_utc`, reconstructs missing HKO
one-decimal contract boundaries, resolves 19a run metadata and
maps 19b identifiers back to the certified target.

The stage stops before constructing the expanded candidate grid.

**Revision v2.** The archived common-support identifier resolver now handles `market_id` without creating duplicate DataFrame column labels. Four identifier routes are tested before the source files are processed.

In [1]:
from __future__ import annotations

import hashlib, json, math, platform, re, sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from IPython.display import display

ROOT=Path.cwd().resolve()
if not (ROOT/'.git').exists():
    raise RuntimeError(f'Run from repository root, not {ROOT}')

STEP='18sA'; UTC=timezone.utc; HKT=ZoneInfo('Asia/Hong_Kong'); EPS=1e-6
RULES=['24h_prior','12h_prior','6h_prior','event_day_open']
RULE_ORDER={r:i for i,r in enumerate(RULES)}
RULE_OFFSET={'24h_prior':-24,'12h_prior':-12,'6h_prior':-6,'event_day_open':0}
OUT=ROOT/'data/processed/18sA_canonical_source_adapters'
REPORT=ROOT/'reports/18sA_canonical_source_adapters'
OUT.mkdir(parents=True,exist_ok=True); REPORT.mkdir(parents=True,exist_ok=True)
EXPECTED={'target':1133,'dates':103,'candidate':4532,'market':4186,'weather':4125,'common':3889,'excluded':643,'groups':355,'books':350,'book_rows':3850}
EXPECTED_RULE={
'24h_prior':(1133,976,1056,921,85,82),
'12h_prior':(1133,1055,1023,978,89,88),
'6h_prior':(1133,1077,1001,967,88,87),
'event_day_open':(1133,1078,1045,1023,93,93)}


def sha(path:Path)->str:
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def norm(s:str)->str: return re.sub(r'[^a-z0-9]+','',s.lower())
def pick(df,aliases,required=True):
    lookup={norm(c):c for c in df.columns}
    for a in aliases:
        if a in df.columns:return a
        if norm(a) in lookup:return lookup[norm(a)]
    if required: raise KeyError(f'None of {aliases} in columns {list(df.columns)}')
    return None

def ids(s): return s.astype('string').str.strip().str.replace(r'\.0$','',regex=True)
def pbool(s,name):
    if pd.api.types.is_bool_dtype(s):return s.astype(bool)
    x=s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
    if x.isna().any(): raise ValueError(f'Cannot parse {name}: {s[x.isna()].drop_duplicates().tolist()}')
    return x.astype(bool)
def read(path): return pd.read_csv(path,low_memory=False,compression='infer')
def candidates(exacts,globs):
    ans=[]
    for x in exacts:
        p=ROOT/x
        if p.is_file():ans.append(p)
    for g in globs: ans += [p for p in ROOT.glob(g) if p.is_file()]
    seen=set(); out=[]
    for p in ans:
        p=p.resolve()
        if p not in seen:seen.add(p);out.append(p)
    return out

def load_valid(label,exacts,globs,validator):
    attempts=[]
    for p in candidates(exacts,globs):
        try:
            df=read(p)
            if validator(df):
                print(f'{label}: {p.relative_to(ROOT)} ({len(df):,} rows)')
                return df,p
            attempts.append(f'{p}: validator false rows={len(df)}')
        except Exception as e: attempts.append(f'{p}: {type(e).__name__}: {e}')
    raise FileNotFoundError(label+' not found\n'+'\n'.join(attempts))

def row_validator(n,groups):
    return lambda df: len(df)==n and all(pick(df,g,False) is not None for g in groups)
def target_validator(n,d):
    def v(df):
        if len(df)!=n:return False
        dc=pick(df,['event_date','date'],False); yc=pick(df,['Y_event_int','realised_yes','y_event','payoff'],False)
        if not dc or not yc:return False
        dates=pd.to_datetime(df[dc],errors='coerce'); y=pd.to_numeric(df[yc],errors='coerce')
        return dates.nunique()==d and not y.isna().any() and pd.DataFrame({'d':dates,'y':y}).groupby('d').y.sum().eq(1).all()
    return v

def etype(x):
    z=norm(str(x)); m={'lower':'lower','lowertail':'lower','lowertailendpoint':'lower','interior':'interior','interiorbin':'interior','upper':'upper','uppertail':'upper'}
    if z not in m:raise ValueError(f'Unknown event type {x}')
    return m[z]
def block(d): return 'march_may_baseline' if d<=pd.Timestamp('2026-05-31') else 'june_external_extension'
def cutoff(d,r):
    t=pd.Timestamp(year=d.year,month=d.month,day=d.day,hour=0,tz=HKT)+pd.Timedelta(hours=RULE_OFFSET[r])
    return t

def member(temp,typ,lo,hi):
    if typ=='lower':return int(temp<float(hi))
    if typ=='interior':return int(float(lo)<=temp<float(hi))
    if typ=='upper':return int(temp>=float(lo))
    raise ValueError(typ)

# Discover verified inputs.
old_target_raw,old_target_src=load_valid('18k target',[
'data/processed/18k_full_hko_contract_event_target_panel.csv','data/processed/18k_full_hko_contract_event_target_panel.csv.gz'],
['data/processed/**/*18k*target*panel*.csv','data/processed/**/*18k*target*panel*.csv.gz'],target_validator(803,73))
june_target_raw,june_target_src=load_valid('18o target',[
'data/processed/18o_june_2026_hko_realised_outcomes/18o_june_2026_contract_outcomes.csv'],
['data/processed/**/*18o*contract*outcomes*.csv'],target_validator(330,30))
old_market_raw,old_market_src=load_valid('18l market',[
'data/processed/18l_full_hko_contract_event_market_scoring_panel.csv','data/processed/18l_full_hko_contract_event_market_scoring_panel.csv.gz'],
['data/processed/**/*18l*market*scoring*panel*.csv','data/processed/**/*18l*market*scoring*panel*.csv.gz'],row_validator(2921,[['event_date'],['market_id'],['decision_rule'],['p_market']]))
june_market_raw,june_market_src=load_valid('18p market',[
'data/processed/18p_june_2026_clob_market_price_recovery/18p_june_2026_market_scoring_panel.csv'],
['data/processed/**/*18p*market*scoring*panel*.csv'],row_validator(1265,[['event_date'],['market_id'],['decision_rule'],['p_market']]))
old_weather_raw,old_weather_src=load_valid('19a weather',[
'data/processed/19a_hko_ecmwf_contract_event_probability_panel.csv','data/processed/19a_hko_ecmwf_contract_event_probability_panel.csv.gz'],
['data/processed/**/*19a*contract*event*probability*panel*.csv','data/processed/**/*19a*contract*event*probability*panel*.csv.gz'],row_validator(3212,[['event_date'],['market_id'],['decision_rule'],['forecast_hko_daily_max_C','forecast_hko_daily_max_c','forecast_daily_max_C','forecast_daily_max_c']]))
june_weather_raw,june_weather_src=load_valid('18q weather',[
'data/processed/18q_june_2026_ecmwf_single_run_forecasts/18q_june_2026_contract_event_membership.csv'],
['data/processed/**/*18q*contract*event*membership*.csv'],row_validator(1320,[['event_date'],['market_id'],['decision_rule'],['forecast_daily_max_c']]))
old_common_raw,old_common_src=load_valid('19b common',[
'data/processed/19b_common_support_market_vs_ecmwf_panel.csv','data/processed/19b_common_support_market_vs_ecmwf_panel.csv.gz','data/processed/19b_common_support_market_vs_ecmwf_proxy_panel.csv','data/processed/19b_common_support_market_vs_ecmwf_proxy_panel.csv.gz'],
['data/processed/**/*19b*common*support*ecmwf*panel*.csv','data/processed/**/*19b*common*support*ecmwf*panel*.csv.gz'],row_validator(2635,[['event_date'],['decision_rule']]))
june_common_raw,june_common_src=load_valid('18r common',[
'data/processed/18r_june_2026_market_weather_common_support/18r_june_2026_common_support_contract_panel.csv'],
['data/processed/**/*18r*common*support*contract*panel*.csv'],row_validator(1254,[['event_date'],['market_id'],['decision_rule']]))



def numeric_alias_series(df,aliases):
    column=pick(df,aliases,False)
    if column is None:
        return pd.Series(np.nan,index=df.index,dtype=float),None
    values=(
        df[column]
        .astype('string')
        .str.strip()
        .replace({
            '':pd.NA,
            'None':pd.NA,
            'none':pd.NA,
            'NA':pd.NA,
            'N/A':pd.NA,
            'nan':pd.NA,
            '-inf':-np.inf,
            'inf':np.inf,
            '+inf':np.inf,
        })
    )
    return pd.to_numeric(values,errors='coerce'),column

def temperature_reference_series(df):
    reference=pd.Series(np.nan,index=df.index,dtype=float)
    used=pd.Series(pd.NA,index=df.index,dtype='string')

    numeric_aliases=[
        'label_value_c','label_value_C','group_item_value',
        'groupItemValue','threshold_c','threshold_C',
        'event_threshold_c','event_threshold_C',
        'temperature_c','temperature_C','contract_temperature_c',
        'contract_temperature_C','strike_c','strike_C','strike',
        'bin_value_c','bin_value_C','value_c','value_C',
    ]

    for alias in numeric_aliases:
        column=pick(df,[alias],False)
        if column is None:
            continue
        values=pd.to_numeric(df[column],errors='coerce')
        mask=reference.isna() & values.notna() & np.isfinite(values)
        reference.loc[mask]=values.loc[mask]
        used.loc[mask]=column

    degree_pattern=r'(-?\d+(?:\.\d+)?)\s*(?:°|º)?\s*[cC](?:\b|_)'
    slug_pattern=r'(?:^|[-_])(-?\d+(?:\.\d+)?)(?:[-_]?(?:c|degc|degrees?c))(?:$|[-_])'
    label_columns=[
        'canonical_label','group_item_title','groupItemTitle',
        'contract_label','label','outcome_label',
    ]
    broader_text_columns=[
        'question','market_question','market_slug','slug','event_slug',
    ]

    for alias in label_columns+broader_text_columns:
        column=pick(df,[alias],False)
        if column is None:
            continue
        text=df[column].astype('string')
        extracted=pd.to_numeric(
            text.str.extract(degree_pattern,expand=False),
            errors='coerce',
        )
        mask=reference.isna() & extracted.notna()
        reference.loc[mask]=extracted.loc[mask]
        used.loc[mask]=column

        if alias in label_columns:
            generic=pd.to_numeric(
                text.str.extract(r'(?<!\d)(-?\d+(?:\.\d+)?)(?!\d)',expand=False),
                errors='coerce',
            )
            mask=reference.isna() & generic.notna()
            reference.loc[mask]=generic.loc[mask]
            used.loc[mask]=column

        if 'slug' in norm(alias):
            slug_value=pd.to_numeric(
                text.str.extract(slug_pattern,expand=False),
                errors='coerce',
            )
            mask=reference.isna() & slug_value.notna()
            reference.loc[mask]=slug_value.loc[mask]
            used.loc[mask]=column

    return reference,used

def event_type_from_text(value):
    text=str(value).strip().lower()
    compact=norm(text)
    direct={
        'lower':'lower',
        'lowertail':'lower',
        'lowertailendpoint':'lower',
        'interior':'interior',
        'interiorbin':'interior',
        'upper':'upper',
        'uppertail':'upper',
    }
    if compact in direct:
        return direct[compact]
    if any(token in text for token in [
        'or below','or lower','or less','below','lower tail',
        'at most','less than or equal',
    ]):
        return 'lower'
    if any(token in text for token in [
        'or higher','or above','or more','higher','upper tail',
        'at least','greater than or equal',
    ]):
        return 'upper'
    if any(token in text for token in [
        'interior','exactly','bin',
    ]):
        return 'interior'
    return pd.NA

def temperature_label(value):
    value=float(value)
    if np.isclose(value,round(value),rtol=0.0,atol=1e-10):
        body=str(int(round(value)))
    else:
        body=f'{value:.6f}'.rstrip('0').rstrip('.')
    return f'{body}°C'

def canonical_target(df,source):
    d=pick(df,['event_date','date'])
    mid=pick(df,['market_id','id','marketId'])
    hko=pick(df,[
        'hko_daily_max_c','hko_daily_max_C',
        'hko_tmax_C','hko_tmax_c',
        'observed_hko_daily_max_c','observed_hko_daily_max_C',
    ])
    y=pick(df,['Y_event_int','realised_yes','realized_yes','y_event','payoff'])

    condition_col=pick(df,['condition_id','conditionId'],False)
    event_id_col=pick(df,['event_id','eventId'],False)
    event_slug_col=pick(df,['event_slug'],False)
    market_slug_col=pick(df,['market_slug','slug'],False)
    question_col=pick(df,['question','market_question'],False)
    label_col=pick(df,[
        'canonical_label','group_item_title','groupItemTitle',
        'contract_label','label','outcome_label',
    ],False)
    yes_token_col=pick(df,[
        'selected_yes_token_id','yes_token_id','yesTokenId',
    ],False)
    no_token_col=pick(df,['no_token_id','noTokenId'],False)

    lower_values,lower_source= numeric_alias_series(df,[
        'lower_bound_c','lower_bound_C','event_lower_C','event_lower_c',
        'event_lower_bound_c','event_lower_bound_C','lower_bound',
        'lower_c','lower_C','bin_lower_c','bin_lower_C',
    ])
    upper_values,upper_source= numeric_alias_series(df,[
        'upper_bound_c','upper_bound_C','event_upper_C','event_upper_c',
        'event_upper_bound_c','event_upper_bound_C','upper_bound',
        'upper_c','upper_C','bin_upper_c','bin_upper_C',
    ])
    reference,reference_source=temperature_reference_series(df)

    type_col=pick(df,[
        'event_type','contract_event_type_v2','contract_event_type',
        'eventType','market_event_type',
    ],False)

    if type_col:
        event_types=df[type_col].map(event_type_from_text)
    else:
        event_types=pd.Series(pd.NA,index=df.index,dtype='string')

    text_candidates=[]
    for column in [
        label_col,question_col,market_slug_col,event_slug_col,
    ]:
        if column is not None:
            text_candidates.append(df[column].astype('string'))

    if text_candidates:
        combined_text=text_candidates[0].fillna('')
        for text in text_candidates[1:]:
            combined_text=combined_text.str.cat(
                text.fillna(''),
                sep=' | ',
            )
        inferred_text_type=combined_text.map(event_type_from_text)
        event_types=event_types.fillna(inferred_text_type)

    bound_type=pd.Series(pd.NA,index=df.index,dtype='string')
    bound_type.loc[
        lower_values.isna() & upper_values.notna()
    ]='lower'
    bound_type.loc[
        lower_values.notna() & upper_values.notna()
    ]='interior'
    bound_type.loc[
        lower_values.notna() & upper_values.isna()
    ]='upper'
    event_types=event_types.fillna(bound_type)

    event_dates=pd.to_datetime(df[d],errors='raise')
    for event_date,index in event_dates.groupby(event_dates).groups.items():
        missing=index[event_types.loc[index].isna()]
        if len(missing)==0:
            continue
        refs=reference.loc[index]
        if refs.notna().sum()==len(index) and refs.nunique()==len(index):
            minimum=refs.idxmin()
            maximum=refs.idxmax()
            event_types.loc[minimum]='lower'
            event_types.loc[maximum]='upper'
            event_types.loc[
                [item for item in index if item not in {minimum,maximum}]
            ]='interior'

    if event_types.isna().any():
        unresolved=df.loc[event_types.isna()].head(20)
        raise AssertionError(
            f'Cannot infer event type for {source}.\n'
            + unresolved.to_string(index=False)
        )

    out=pd.DataFrame({
        'event_date':event_dates,
        'market_id':ids(df[mid]),
        'event_type':event_types.astype(str),
        'lower_bound_c':lower_values.astype(float),
        'upper_bound_c':upper_values.astype(float),
        'bound_reference_c':reference.astype(float),
        'hko_daily_max_c':pd.to_numeric(df[hko],errors='raise'),
        'Y_event_int':pd.to_numeric(df[y],errors='raise').astype(int),
    })

    out['condition_id']=(
        ids(df[condition_col])
        if condition_col
        else pd.Series(pd.NA,index=df.index,dtype='string')
    )
    out['event_id']=(
        ids(df[event_id_col])
        if event_id_col
        else pd.Series(pd.NA,index=df.index,dtype='string')
    )
    out['event_slug']=(
        df[event_slug_col].astype('string')
        if event_slug_col
        else pd.Series(pd.NA,index=df.index,dtype='string')
    )
    out['market_slug']=(
        df[market_slug_col].astype('string')
        if market_slug_col
        else pd.Series(pd.NA,index=df.index,dtype='string')
    )
    out['question']=(
        df[question_col].astype('string')
        if question_col
        else pd.Series(pd.NA,index=df.index,dtype='string')
    )
    out['canonical_label']=(
        df[label_col].astype('string')
        if label_col
        else pd.Series(pd.NA,index=df.index,dtype='string')
    )
    out['selected_yes_token_id']=(
        ids(df[yes_token_col])
        if yes_token_col
        else pd.Series(pd.NA,index=df.index,dtype='string')
    )
    out['no_token_id']=(
        ids(df[no_token_col])
        if no_token_col
        else pd.Series(pd.NA,index=df.index,dtype='string')
    )

    original_lower=out['lower_bound_c'].copy()
    original_upper=out['upper_bound_c'].copy()

    # Standardise irrelevant infinite bounds to missing.
    out.loc[out['event_type'].eq('lower'),'lower_bound_c']=np.nan
    out.loc[out['event_type'].eq('upper'),'upper_bound_c']=np.nan

    interior=out['event_type'].eq('interior')
    missing_lower=interior & out['lower_bound_c'].isna()
    out.loc[missing_lower,'lower_bound_c']=out.loc[
        missing_lower,'bound_reference_c'
    ]

    missing_upper=interior & out['upper_bound_c'].isna()
    out.loc[
        missing_upper & out['lower_bound_c'].notna(),
        'upper_bound_c',
    ]=out.loc[
        missing_upper & out['lower_bound_c'].notna(),
        'lower_bound_c',
    ]+1.0

    still_missing_lower=interior & out['lower_bound_c'].isna()
    out.loc[
        still_missing_lower & out['upper_bound_c'].notna(),
        'lower_bound_c',
    ]=out.loc[
        still_missing_lower & out['upper_bound_c'].notna(),
        'upper_bound_c',
    ]-1.0

    # Enforce the verified one-degree interior-bin definition.
    interior_resolved=interior & out['lower_bound_c'].notna()
    out.loc[
        interior_resolved,
        'upper_bound_c',
    ]=out.loc[
        interior_resolved,
        'lower_bound_c',
    ]+1.0

    # Endpoint bounds are defined by the adjacent interior bins.
    for event_date,index in out.groupby('event_date').groups.items():
        book=out.loc[index]
        counts=book['event_type'].value_counts().to_dict()
        if counts!={'interior':9,'lower':1,'upper':1}:
            raise AssertionError(
                f'{source} {event_date.date()}: '
                f'expected 1 lower, 9 interior and 1 upper; found {counts}'
            )

        interior_rows=book.loc[
            book['event_type'].eq('interior')
        ].sort_values('lower_bound_c')

        if interior_rows[
            ['lower_bound_c','upper_bound_c']
        ].isna().any().any():
            raise AssertionError(
                f'{source} {event_date.date()}: '
                'interior bounds remain unresolved'
            )

        lower_values_sorted=interior_rows[
            'lower_bound_c'
        ].to_numpy(dtype=float)
        upper_values_sorted=interior_rows[
            'upper_bound_c'
        ].to_numpy(dtype=float)

        if not np.isclose(
            upper_values_sorted-lower_values_sorted,
            1.0,
            rtol=0.0,
            atol=1e-10,
        ).all():
            raise AssertionError(
                f'{source} {event_date.date()}: '
                'interior bins are not one degree wide'
            )

        if not np.isclose(
            np.diff(lower_values_sorted),
            1.0,
            rtol=0.0,
            atol=1e-10,
        ).all():
            raise AssertionError(
                f'{source} {event_date.date()}: '
                'interior bins are not contiguous'
            )

        lower_index=book.index[
            book['event_type'].eq('lower')
        ][0]
        upper_index=book.index[
            book['event_type'].eq('upper')
        ][0]

        out.loc[
            lower_index,
            'upper_bound_c',
        ]=float(lower_values_sorted.min())
        out.loc[
            upper_index,
            'lower_bound_c',
        ]=float(upper_values_sorted.max())

    required_missing=(
        (
            out['event_type'].eq('lower')
            & out['upper_bound_c'].isna()
        )
        | (
            out['event_type'].eq('interior')
            & (
                out['lower_bound_c'].isna()
                | out['upper_bound_c'].isna()
            )
        )
        | (
            out['event_type'].eq('upper')
            & out['lower_bound_c'].isna()
        )
    )
    if required_missing.any():
        raise AssertionError(
            f'Missing canonical bounds remain for {source}'
        )

    label_missing=(
        out['canonical_label'].isna()
        | out['canonical_label'].astype(str).str.strip().isin(
            ['', '<NA>', 'nan', 'None']
        )
    )

    for index,row in out.loc[label_missing].iterrows():
        if row['event_type']=='lower':
            label_value=float(row['upper_bound_c'])-1.0
            label=f'{temperature_label(label_value)} or below'
        elif row['event_type']=='interior':
            label=temperature_label(row['lower_bound_c'])
        else:
            label=f'{temperature_label(row["lower_bound_c"])} or higher'
        out.loc[index,'canonical_label']=label

    lower_changed=~np.isclose(
        original_lower,
        out['lower_bound_c'],
        rtol=0.0,
        atol=1e-10,
        equal_nan=True,
    )
    upper_changed=~np.isclose(
        original_upper,
        out['upper_bound_c'],
        rtol=0.0,
        atol=1e-10,
        equal_nan=True,
    )
    out['bound_reconstruction_applied']=(
        lower_changed | upper_changed
    )
    out['bound_reference_source']=reference_source.astype('string')
    out['source_lower_bound_column']=(
        lower_source if lower_source else ''
    )
    out['source_upper_bound_column']=(
        upper_source if upper_source else ''
    )

    out['Y_no_int']=1-out['Y_event_int']
    out['contract_event_type']=out['event_type'].map({
        'lower':'lower_tail_endpoint',
        'interior':'interior_bin',
        'upper':'upper_tail',
    })
    out['sample_block']=out['event_date'].map(block)
    out['final_modelling_split']='UNASSIGNED'
    out['outcome_history_admissibility_status']='NOT_YET_APPLIED'
    out['source_block']=source

    if out['market_id'].isna().any():
        raise AssertionError(f'{source}: missing market identifiers')
    if out.duplicated(['event_date','market_id']).any():
        raise AssertionError(f'{source}: duplicate target keys')
    if not out.groupby('event_date')['market_id'].size().eq(11).all():
        raise AssertionError(f'{source}: incomplete target books')
    if not out.groupby('event_date')['Y_event_int'].sum().eq(1).all():
        raise AssertionError(f'{source}: target books lack one winner')
    if not out['Y_event_int'].isin([0,1]).all():
        raise AssertionError(f'{source}: non-binary outcomes')

    print(
        f'{source}: canonical bounds ready; '
        f'reconstructed rows={int(out["bound_reconstruction_applied"].sum())}'
    )

    return out


old_target=canonical_target(old_target_raw,'march_may_baseline'); june_target=canonical_target(june_target_raw,'june_external_extension')
target=pd.concat([old_target,june_target],ignore_index=True).sort_values(['event_date','market_id']).reset_index(drop=True)
if len(target)!=1133 or target.event_date.nunique()!=103 or target.market_id.nunique()!=1133:raise AssertionError('Target totals')
if not target.groupby('event_date').market_id.size().eq(11).all():raise AssertionError('Target book sizes')
if not target.groupby('event_date').Y_event_int.sum().eq(1).all():raise AssertionError('Target winner counts')
META=['event_date','market_id','condition_id','event_id','event_slug','market_slug','question','canonical_label','event_type','contract_event_type','lower_bound_c','upper_bound_c','bound_reference_c','bound_reconstruction_applied','bound_reference_source','source_lower_bound_column','source_upper_bound_column','selected_yes_token_id','no_token_id','hko_daily_max_c','Y_event_int','Y_no_int','sample_block','final_modelling_split','outcome_history_admissibility_status']



def parse_datetime_aliases(
    df,
    *,
    text_aliases,
    unix_aliases,
    required,
    label,
):
    text_column=pick(df,text_aliases,False)
    if text_column is not None:
        parsed=pd.to_datetime(
            df[text_column],
            utc=True,
            errors='coerce',
        )
        if parsed.notna().all():
            return parsed,text_column

    unix_column=pick(df,unix_aliases,False)
    if unix_column is not None:
        numeric=pd.to_numeric(
            df[unix_column],
            errors='coerce',
        )
        finite=numeric.dropna()
        if not finite.empty:
            magnitude=float(finite.abs().median())
            unit='ms' if magnitude>=1e11 else 's'
            parsed=pd.to_datetime(
                numeric,
                unit=unit,
                utc=True,
                errors='coerce',
            )
            if parsed.notna().all():
                return parsed,unix_column

    if required:
        raise KeyError(
            f'Could not resolve {label}. '
            f'Text aliases={text_aliases}; '
            f'Unix aliases={unix_aliases}; '
            f'columns={list(df.columns)}'
        )

    return pd.Series(pd.NaT,index=df.index,dtype='datetime64[ns, UTC]'),None

def canonical_market(df,source,targ):
    d=pick(df,['event_date','date'])
    mid=pick(df,['market_id','marketId'])
    r=pick(df,['decision_rule','rule'])
    p=pick(df,[
        'p_market','market_probability','selected_price',
        'yes_price','price',
    ])

    event_date=pd.to_datetime(df[d],errors='raise')
    decision_rule=df[r].astype(str)

    cutoff_values,cutoff_source=parse_datetime_aliases(
        df,
        text_aliases=[
            'decision_cutoff_utc','decision_time_utc',
            'decision_timestamp_utc','cutoff_utc',
        ],
        unix_aliases=[
            'decision_cutoff_unix','decision_time_unix',
            'decision_timestamp_unix','cutoff_unix',
        ],
        required=False,
        label=f'{source} decision cutoff',
    )
    if cutoff_values.isna().any():
        derived=pd.Series(
            [
                cutoff(date,rule).tz_convert('UTC')
                for date,rule in zip(event_date,decision_rule)
            ],
            index=df.index,
            dtype='datetime64[ns, UTC]',
        )
        cutoff_values=cutoff_values.fillna(derived)

    price_time_values,price_time_source=parse_datetime_aliases(
        df,
        text_aliases=[
            'selected_price_timestamp_utc',
            'decision_price_timestamp_utc',
            'selected_record_timestamp_utc',
            'price_timestamp_utc','record_timestamp_utc',
            'selected_timestamp_utc','timestamp_utc',
        ],
        unix_aliases=[
            'selected_price_timestamp_unix',
            'decision_price_timestamp_unix',
            'selected_record_timestamp_unix',
            'price_timestamp_unix','record_timestamp_unix',
            'selected_timestamp_unix','timestamp',
        ],
        required=True,
        label=f'{source} selected market timestamp',
    )

    out=pd.DataFrame({
        'event_date':event_date,
        'market_id':ids(df[mid]),
        'decision_rule':decision_rule,
        'p_market':pd.to_numeric(df[p],errors='raise'),
        'decision_cutoff_utc':cutoff_values,
        'selected_price_timestamp_utc':price_time_values,
    })

    out['price_staleness_hours']=(
        out['decision_cutoff_utc']
        -out['selected_price_timestamp_utc']
    ).dt.total_seconds()/3600.0
    out['source_cutoff_column']=cutoff_source or 'derived_from_rule'
    out['source_price_timestamp_column']=price_time_source or ''

    out=out.merge(
        targ[META],
        on=['event_date','market_id'],
        how='left',
        validate='many_to_one',
    )
    out['source_block']=source
    out['decision_rule_order']=out['decision_rule'].map(RULE_ORDER)

    out['market_binary_brier']=(
        out['p_market']-out['Y_event_int']
    )**2
    clipped=out['p_market'].clip(EPS,1-EPS)
    out['market_binary_log_score']=-(
        out['Y_event_int']*np.log(clipped)
        +(1-out['Y_event_int'])*np.log(1-clipped)
    )

    if out['decision_rule_order'].isna().any():
        raise AssertionError(f'{source}: unexpected decision rule')
    if out[
        ['canonical_label','event_type','Y_event_int']
    ].isna().any().any():
        raise AssertionError(
            f'{source}: market rows failed to join target metadata'
        )
    if out.duplicated(
        ['event_date','market_id','decision_rule']
    ).any():
        raise AssertionError(f'{source}: duplicate market keys')
    if not out['p_market'].between(0,1).all():
        raise AssertionError(f'{source}: invalid market probabilities')
    if not (
        out['selected_price_timestamp_utc']
        <=out['decision_cutoff_utc']
    ).all():
        bad=out.loc[
            out['selected_price_timestamp_utc']
            >out['decision_cutoff_utc']
        ].head(20)
        raise AssertionError(
            f'{source}: market look-ahead violation\n'
            +bad.to_string(index=False)
        )
    if (out['price_staleness_hours']<-1e-10).any():
        raise AssertionError(f'{source}: negative market staleness')

    return out


old_market=canonical_market(old_market_raw,'march_may_baseline',old_target);june_market=canonical_market(june_market_raw,'june_external_extension',june_target)
market=pd.concat([old_market,june_market],ignore_index=True).sort_values(['event_date','decision_rule_order','market_id']).reset_index(drop=True)
if len(market)!=4186:raise AssertionError(f'Market rows {len(market)}')



def canonical_weather(df,source,targ,expected):
    d=pick(df,['event_date','date'])
    mid=pick(df,['market_id','marketId'])
    r=pick(df,['decision_rule','rule'])
    fm=pick(df,[
        'forecast_hko_daily_max_C','forecast_hko_daily_max_c',
        'forecast_daily_max_c','forecast_daily_max_C',
        'forecast_max_c','forecast_max_C',
        'predicted_daily_max_c','predicted_daily_max_C',
    ])

    event_date=pd.to_datetime(df[d],errors='raise')
    decision_rule=df[r].astype(str)
    forecast_max=pd.to_numeric(df[fm],errors='coerce')

    flag=pick(df,[
        'forecast_path_ready','probability_ready',
        'weather_path_ready','ready','daily_max_ready',
    ],False)
    if flag:
        ready=pbool(df[flag],f'{source} readiness')
        ready=ready & forecast_max.notna()
    else:
        ready=forecast_max.notna()

    cutoff_values,cutoff_source=parse_datetime_aliases(
        df,
        text_aliases=[
            'decision_cutoff_utc','decision_time_utc',
            'decision_timestamp_utc','cutoff_utc',
        ],
        unix_aliases=[
            'decision_cutoff_unix','decision_time_unix',
            'decision_timestamp_unix','cutoff_unix',
        ],
        required=False,
        label=f'{source} decision cutoff',
    )
    if cutoff_values.isna().any():
        derived=pd.Series(
            [
                cutoff(date,rule).tz_convert('UTC')
                for date,rule in zip(event_date,decision_rule)
            ],
            index=df.index,
            dtype='datetime64[ns, UTC]',
        )
        cutoff_values=cutoff_values.fillna(derived)

    run_initialisation,run_initialisation_source=parse_datetime_aliases(
        df,
        text_aliases=[
            'selected_run_initialisation_utc',
            'selected_run_initialization_utc',
            'selected_run_init_utc',
            'run_initialisation_utc',
            'run_initialization_utc',
            'forecast_run_initialisation_utc',
            'forecast_run_initialization_utc',
            'selected_run_utc','run_time_utc','issue_time_utc',
        ],
        unix_aliases=[
            'selected_run_initialisation_unix',
            'selected_run_initialization_unix',
            'selected_run_init_unix','run_initialisation_unix',
            'run_initialization_unix','run_time_unix',
            'issue_time_unix',
        ],
        required=False,
        label=f'{source} run initialisation',
    )
    run_available,run_available_source=parse_datetime_aliases(
        df,
        text_aliases=[
            'selected_run_available_utc',
            'selected_run_available_utc_assumed',
            'run_available_utc','available_time_utc',
            'forecast_available_utc',
        ],
        unix_aliases=[
            'selected_run_available_unix',
            'run_available_unix','available_time_unix',
            'forecast_available_unix',
        ],
        required=False,
        label=f'{source} run availability',
    )

    missing_init=run_initialisation.isna() & run_available.notna()
    run_initialisation.loc[missing_init]=(
        run_available.loc[missing_init]-pd.Timedelta(hours=6)
    )

    missing_available=run_available.isna() & run_initialisation.notna()
    run_available.loc[missing_available]=(
        run_initialisation.loc[missing_available]+pd.Timedelta(hours=6)
    )

    both_missing=run_initialisation.isna() & run_available.isna()
    if both_missing.any():
        reconstructed=(
            cutoff_values.loc[both_missing]
            -pd.Timedelta(hours=6)
        ).dt.floor('6h')
        run_initialisation.loc[both_missing]=reconstructed
        run_available.loc[both_missing]=(
            reconstructed+pd.Timedelta(hours=6)
        )

    run_key_column=pick(df,[
        'selected_run_key','run_key','forecast_run_key',
    ],False)

    out=pd.DataFrame({
        'event_date':event_date,
        'market_id':ids(df[mid]),
        'decision_rule':decision_rule,
        'weather_path_ready':ready,
        'forecast_daily_max_c':forecast_max,
        'decision_cutoff_utc':cutoff_values,
        'selected_run_initialisation_utc':run_initialisation,
        'selected_run_available_utc':run_available,
    })

    if run_key_column:
        out['selected_run_key']=df[run_key_column].astype('string')
    else:
        out['selected_run_key']=out[
            'selected_run_initialisation_utc'
        ].dt.strftime('%Y%m%dT%H%MZ')

    out['source_forecast_max_column']=fm
    out['source_cutoff_column']=cutoff_source or 'derived_from_rule'
    out['source_run_initialisation_column']=(
        run_initialisation_source or 'derived'
    )
    out['source_run_available_column']=(
        run_available_source or 'derived'
    )
    out['run_metadata_reconstructed']=both_missing

    out=out.loc[out['weather_path_ready']].copy()

    if out[
        [
            'forecast_daily_max_c',
            'selected_run_initialisation_utc',
            'selected_run_available_utc',
        ]
    ].isna().any().any():
        raise AssertionError(
            f'{source}: ready weather rows have missing core fields'
        )

    out=out.merge(
        targ[META],
        on=['event_date','market_id'],
        how='left',
        validate='many_to_one',
    )
    out['source_block']=source
    out['decision_rule_order']=out['decision_rule'].map(RULE_ORDER)
    out['forecast_error_c']=(
        out['forecast_daily_max_c']-out['hko_daily_max_c']
    )
    out['absolute_error_c']=out['forecast_error_c'].abs()
    out['deterministic_forecast_event_indicator']=out.apply(
        lambda row:member(
            float(row['forecast_daily_max_c']),
            row['event_type'],
            row['lower_bound_c'],
            row['upper_bound_c'],
        ),
        axis=1,
    ).astype(int)

    if len(out)!=expected:
        raise AssertionError(
            f'{source}: expected {expected} ready weather rows, '
            f'found {len(out)}'
        )
    if out['decision_rule_order'].isna().any():
        raise AssertionError(f'{source}: unexpected decision rule')
    if out[
        ['canonical_label','event_type','hko_daily_max_c']
    ].isna().any().any():
        raise AssertionError(
            f'{source}: weather rows failed to join target metadata'
        )
    if out.duplicated(
        ['event_date','market_id','decision_rule']
    ).any():
        raise AssertionError(f'{source}: duplicate weather keys')
    if not np.isfinite(out['forecast_daily_max_c']).all():
        raise AssertionError(f'{source}: non-finite forecast maximum')
    if not (
        out['selected_run_available_utc']
        <=out['decision_cutoff_utc']
    ).all():
        bad=out.loc[
            out['selected_run_available_utc']
            >out['decision_cutoff_utc']
        ].head(20)
        raise AssertionError(
            f'{source}: run unavailable at cutoff\n'
            +bad.to_string(index=False)
        )
    if not set(
        out['selected_run_initialisation_utc'].dt.hour.unique()
    ).issubset({0,6,12,18}):
        raise AssertionError(
            f'{source}: invalid run cycle hour'
        )

    group_columns=['event_date','decision_rule']
    group_size=out.groupby(group_columns)['market_id'].size()
    if not group_size.eq(11).all():
        raise AssertionError(
            f'{source}: a ready weather path is not a full book'
        )

    forecast_nunique=out.groupby(
        group_columns
    )['forecast_daily_max_c'].nunique(dropna=False)
    run_nunique=out.groupby(
        group_columns
    )['selected_run_initialisation_utc'].nunique(dropna=False)
    if not forecast_nunique.eq(1).all():
        raise AssertionError(
            f'{source}: inconsistent forecast maximum within a book'
        )
    if not run_nunique.eq(1).all():
        raise AssertionError(
            f'{source}: inconsistent selected run within a book'
        )

    if not out.groupby(
        group_columns
    )['deterministic_forecast_event_indicator'].sum().eq(1).all():
        raise AssertionError(
            f'{source}: deterministic forecast does not select '
            'exactly one contract'
        )

    return out


old_weather=canonical_weather(old_weather_raw,'march_may_baseline',old_target,2816);june_weather=canonical_weather(june_weather_raw,'june_external_extension',june_target,1309)
weather=pd.concat([old_weather,june_weather],ignore_index=True).sort_values(['event_date','decision_rule_order','market_id']).reset_index(drop=True)
if len(weather)!=4125:raise AssertionError('Weather total')

KEY=['event_date','market_id','decision_rule']

def identifier_candidates(df,aliases):
    alias_norms={norm(alias) for alias in aliases}
    ranked=[]
    for column in df.columns:
        column_norm=norm(column)
        score=None
        if column_norm in alias_norms:
            score=0
        elif any(
            column_norm.startswith(alias_norm)
            or column_norm.endswith(alias_norm)
            for alias_norm in alias_norms
        ):
            score=1
        elif any(alias_norm in column_norm for alias_norm in alias_norms):
            score=2
        if score is not None:
            ranked.append((score,len(column_norm),column))
    return [
        column
        for _,_,column in sorted(
            ranked,
            key=lambda item:(item[0],item[1],item[2]),
        )
    ]

def mapped_common_keyset(df,target_df,label):
    """
    Resolve an archived merged-panel identifier back to the canonical
    date-market-rule key.

    The market-ID case is handled separately because selecting
    ['event_date', 'market_id', 'market_id'] would create duplicate
    column labels and make mapping['market_id'] a DataFrame rather than
    a Series.
    """
    date_column=pick(df,['event_date','date'])
    rule_column=pick(df,['decision_rule'])

    base=pd.DataFrame({
        'event_date':pd.to_datetime(
            df[date_column],
            errors='raise',
        ),
        'decision_rule':df[rule_column].astype(str),
    })

    concepts=[
        (
            'market_id',
            [
                'market_id',
                'market_id_market',
                'market_id_x',
                'market_id_left',
                'polymarket_market_id',
                'contract_market_id',
            ],
            'market_id',
        ),
        (
            'condition_id',
            [
                'condition_id',
                'condition_id_market',
                'condition_id_x',
                'condition_id_left',
                'conditionId',
            ],
            'condition_id',
        ),
        (
            'selected_yes_token_id',
            [
                'selected_yes_token_id',
                'yes_token_id',
                'yes_token_id_market',
                'token_id',
                'asset_id',
            ],
            'selected_yes_token_id',
        ),
        (
            'market_slug',
            [
                'market_slug',
                'market_slug_market',
                'market_slug_x',
                'slug',
            ],
            'market_slug',
        ),
    ]

    attempts=[]

    for concept,aliases,target_column in concepts:
        if target_column not in target_df.columns:
            attempts.append(
                f'{concept}:target_column_missing'
            )
            continue

        if target_column == 'market_id':
            mapping=target_df[
                ['event_date','market_id']
            ].copy()
            mapping['_identifier']=ids(
                mapping['market_id']
            )
            mapping['market_id']=ids(
                mapping['market_id']
            )
        else:
            mapping=target_df[
                ['event_date','market_id',target_column]
            ].copy()
            mapping['market_id']=ids(
                mapping['market_id']
            )
            mapping['_identifier']=ids(
                mapping[target_column]
            )
            mapping=mapping.drop(
                columns=[target_column]
            )

        mapping['event_date']=pd.to_datetime(
            mapping['event_date'],
            errors='raise',
        )
        mapping=mapping.dropna(
            subset=['_identifier','market_id']
        ).drop_duplicates()

        if mapping.empty:
            attempts.append(
                f'{concept}:canonical_mapping_empty'
            )
            continue

        duplicate_mapping=(
            mapping.groupby(
                ['event_date','_identifier']
            )['market_id']
            .nunique()
            .gt(1)
        )

        if duplicate_mapping.any():
            examples=(
                duplicate_mapping.loc[
                    duplicate_mapping
                ]
                .head(10)
                .index
                .tolist()
            )
            attempts.append(
                f'{concept}:canonical_mapping_not_unique='
                f'{examples}'
            )
            continue

        mapping=mapping.drop_duplicates(
            ['event_date','_identifier']
        )

        source_columns=identifier_candidates(
            df,
            aliases,
        )

        if not source_columns:
            attempts.append(
                f'{concept}:source_column_missing'
            )
            continue

        for column in source_columns:
            source_values=df[column]

            if isinstance(source_values,pd.DataFrame):
                attempts.append(
                    f'{concept}:{column}:duplicate_source_header'
                )
                continue

            trial=base.copy()
            trial['_identifier']=ids(
                source_values
            )

            joined=trial.merge(
                mapping,
                on=['event_date','_identifier'],
                how='left',
                validate='many_to_one',
            )

            missing=int(
                joined['market_id'].isna().sum()
            )
            if missing:
                attempts.append(
                    f'{concept}:{column}:unmapped={missing}'
                )
                continue

            keys=joined[
                [
                    'event_date',
                    'market_id',
                    'decision_rule',
                ]
            ].copy()

            duplicate_keys=int(
                keys.duplicated().sum()
            )
            if duplicate_keys:
                attempts.append(
                    f'{concept}:{column}:duplicate_keys='
                    f'{duplicate_keys}'
                )
                continue

            print(
                f'{label} identifier resolved with '
                f'{concept} column: {column}'
            )

            return set(
                keys.itertuples(
                    index=False,
                    name=None,
                )
            )

    raise AssertionError(
        f'{label}: no archived identifier could be mapped '
        'uniquely to the canonical target. '
        f'Available columns={list(df.columns)}. '
        f'Attempts={attempts}'
    )


def _self_test_mapped_common_keyset():
    target_test=pd.DataFrame({
        'event_date':pd.to_datetime([
            '2026-03-01',
            '2026-03-01',
        ]),
        'market_id':['101','102'],
        'condition_id':['c101','c102'],
        'selected_yes_token_id':['t101','t102'],
        'market_slug':['m101','m102'],
    })

    expected={
        (
            pd.Timestamp('2026-03-01'),
            '101',
            '24h_prior',
        ),
        (
            pd.Timestamp('2026-03-01'),
            '102',
            '24h_prior',
        ),
    }

    source_variants=[
        pd.DataFrame({
            'event_date':['2026-03-01','2026-03-01'],
            'decision_rule':['24h_prior','24h_prior'],
            'market_id_market':['101','102'],
        }),
        pd.DataFrame({
            'event_date':['2026-03-01','2026-03-01'],
            'decision_rule':['24h_prior','24h_prior'],
            'condition_id_x':['c101','c102'],
        }),
        pd.DataFrame({
            'event_date':['2026-03-01','2026-03-01'],
            'decision_rule':['24h_prior','24h_prior'],
            'yes_token_id_market':['t101','t102'],
        }),
        pd.DataFrame({
            'event_date':['2026-03-01','2026-03-01'],
            'decision_rule':['24h_prior','24h_prior'],
            'market_slug_x':['m101','m102'],
        }),
    ]

    for position,source_test in enumerate(
        source_variants,
        start=1,
    ):
        actual=mapped_common_keyset(
            source_test,
            target_test,
            f'self-test-{position}',
        )
        if actual!=expected:
            raise AssertionError(
                f'mapped_common_keyset self-test '
                f'{position} failed'
            )

    print(
        'mapped_common_keyset four-route self-test: PASS'
    )


_self_test_mapped_common_keyset()


old_inter=(
    set(old_market[KEY].itertuples(index=False,name=None))
    & set(old_weather[KEY].itertuples(index=False,name=None))
)
june_inter=(
    set(june_market[KEY].itertuples(index=False,name=None))
    & set(june_weather[KEY].itertuples(index=False,name=None))
)

old_archived_keys=mapped_common_keyset(
    old_common_raw,
    old_target,
    '19b common support',
)
june_archived_keys=mapped_common_keyset(
    june_common_raw,
    june_target,
    '18r common support',
)

if old_inter!=old_archived_keys or len(old_inter)!=2635:
    raise AssertionError('19b key crosscheck')
if june_inter!=june_archived_keys or len(june_inter)!=1254:
    raise AssertionError('18r key crosscheck')

# ------------------------------------------------------------------
# Freeze canonical source adapters before aggregation.
# ------------------------------------------------------------------

target_output = target.copy()
market_output = market.copy()
weather_output = weather.copy()

for frame in [target_output, market_output, weather_output]:
    frame["event_date"] = pd.to_datetime(
        frame["event_date"]
    ).dt.date.astype(str)

    for column in frame.columns:
        lowered = column.lower()
        if (
            "timestamp" in lowered
            or lowered.endswith("_utc")
            or lowered.endswith("_hkt")
            or "initialisation" in lowered
            or "initialization" in lowered
            or "available" in lowered
        ):
            frame[column] = frame[column].astype(str)

target_path = OUT / "18sA_canonical_contract_outcome_panel.csv"
market_path = OUT / "18sA_canonical_market_panel.csv"
weather_path = OUT / "18sA_canonical_weather_panel.csv"

target_output.to_csv(target_path, index=False)
market_output.to_csv(market_path, index=False)
weather_output.to_csv(weather_path, index=False)

source_rows = []
for role, path, frame in [
    ("march_may_target", old_target_src, old_target_raw),
    ("june_target", june_target_src, june_target_raw),
    ("march_may_market", old_market_src, old_market_raw),
    ("june_market", june_market_src, june_market_raw),
    ("march_may_weather", old_weather_src, old_weather_raw),
    ("june_weather", june_weather_src, june_weather_raw),
    (
        "march_may_common_support_crosscheck",
        old_common_src,
        old_common_raw,
    ),
    (
        "june_common_support_crosscheck",
        june_common_src,
        june_common_raw,
    ),
]:
    source_rows.append(
        {
            "input_role": role,
            "path": str(path.relative_to(ROOT)),
            "rows": len(frame),
            "sha256": sha(path),
        }
    )

source_inventory = pd.DataFrame(source_rows)
source_inventory_path = OUT / "18sA_source_inventory.csv"
source_inventory.to_csv(source_inventory_path, index=False)

schema_resolution = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "revision": "v1",
    "target": {
        "rows": int(len(target)),
        "dates": int(target["event_date"].nunique()),
        "march_may_reconstructed_bound_rows": int(
            old_target["bound_reconstruction_applied"].sum()
        ),
        "june_reconstructed_bound_rows": int(
            june_target["bound_reconstruction_applied"].sum()
        ),
        "bound_reference_sources": sorted(
            {
                str(value)
                for value in target[
                    "bound_reference_source"
                ].dropna().unique()
            }
        ),
    },
    "market": {
        "rows": int(len(market)),
        "selected_timestamp_source_columns": sorted(
            {
                str(value)
                for value in market[
                    "source_price_timestamp_column"
                ].dropna().unique()
            }
        ),
        "decision_cutoff_source_columns": sorted(
            {
                str(value)
                for value in market[
                    "source_cutoff_column"
                ].dropna().unique()
            }
        ),
    },
    "weather": {
        "rows": int(len(weather)),
        "forecast_max_source_columns": sorted(
            {
                str(value)
                for value in weather[
                    "source_forecast_max_column"
                ].dropna().unique()
            }
        ),
        "run_initialisation_source_columns": sorted(
            {
                str(value)
                for value in weather[
                    "source_run_initialisation_column"
                ].dropna().unique()
            }
        ),
        "run_available_source_columns": sorted(
            {
                str(value)
                for value in weather[
                    "source_run_available_column"
                ].dropna().unique()
            }
        ),
        "reconstructed_run_metadata_rows": int(
            weather["run_metadata_reconstructed"].sum()
        ),
    },
    "common_support_crosschecks": {
        "march_may_exact_keys": int(len(old_inter)),
        "june_exact_keys": int(len(june_inter)),
        "march_may_matches_19b": bool(
            old_inter == old_archived_keys
        ),
        "june_matches_18r": bool(
            june_inter == june_archived_keys
        ),
    },
    "probability_bridge_retained": False,
    "final_modelling_split_assigned": False,
    "outcome_availability_history_applied": False,
}

schema_resolution_path = OUT / "18sA_schema_resolution.json"
schema_resolution_path.write_text(
    json.dumps(schema_resolution, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

checks = pd.DataFrame(
    [
        {
            "check": "canonical_target_rows",
            "passed": len(target) == 1133,
            "detail": f"rows={len(target)}",
            "blocking": True,
        },
        {
            "check": "canonical_target_dates",
            "passed": target["event_date"].nunique() == 103,
            "detail": (
                f"dates={target['event_date'].nunique()}"
            ),
            "blocking": True,
        },
        {
            "check": "canonical_market_rows",
            "passed": len(market) == 4186,
            "detail": f"rows={len(market)}",
            "blocking": True,
        },
        {
            "check": "canonical_weather_rows",
            "passed": len(weather) == 4125,
            "detail": f"rows={len(weather)}",
            "blocking": True,
        },
        {
            "check": "march_may_common_keys_equal_19b",
            "passed": old_inter == old_archived_keys,
            "detail": f"keys={len(old_inter)}",
            "blocking": True,
        },
        {
            "check": "june_common_keys_equal_18r",
            "passed": june_inter == june_archived_keys,
            "detail": f"keys={len(june_inter)}",
            "blocking": True,
        },
        {
            "check": "market_no_lookahead",
            "passed": (
                market["selected_price_timestamp_utc"]
                <= market["decision_cutoff_utc"]
            ).all(),
            "detail": "all market timestamps at or before cut-off",
            "blocking": True,
        },
        {
            "check": "weather_run_admissibility",
            "passed": (
                weather["selected_run_available_utc"]
                <= weather["decision_cutoff_utc"]
            ).all(),
            "detail": "all weather runs available by cut-off",
            "blocking": True,
        },
        {
            "check": "no_gaussian_bridge_columns",
            "passed": not any(
                (
                    "gaussian" in column.lower()
                    or "ecmwf_proxy" in column.lower()
                    or column.lower().startswith("sigma")
                )
                for frame in [target, market, weather]
                for column in frame.columns
            ),
            "detail": "canonical adapter outputs only",
            "blocking": True,
        },
    ]
)

if not checks["passed"].all():
    raise AssertionError(
        "18sA blocking checks failed:\n"
        + checks.loc[
            ~checks["passed"]
        ].to_string(index=False)
    )

checks_path = OUT / "18sA_integrity_checks.csv"
checks.to_csv(checks_path, index=False)

issues = pd.DataFrame(
    columns=[
        "issue_level",
        "issue_code",
        "event_date",
        "market_id",
        "decision_rule",
        "detail",
        "blocking",
    ]
)
issues_path = OUT / "18sA_issues.csv"
issues.to_csv(issues_path, index=False)

summary = {
    "step": "18sA",
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "canonical_target_rows": int(len(target)),
    "canonical_target_dates": int(
        target["event_date"].nunique()
    ),
    "canonical_market_rows": int(len(market)),
    "canonical_weather_rows": int(len(weather)),
    "march_may_common_keys": int(len(old_inter)),
    "june_common_keys": int(len(june_inter)),
    "march_may_common_keys_equal_19b": bool(
        old_inter == old_archived_keys
    ),
    "june_common_keys_equal_18r": bool(
        june_inter == june_archived_keys
    ),
    "probability_bridge_retained": False,
    "final_modelling_split_assigned": False,
    "outcome_availability_history_applied": False,
    "issue_rows": 0,
}

summary_path = OUT / "18sA_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

report_lines = [
    "# 18sA canonical source adapters",
    "",
    "**PASS**",
    "",
    f"- Canonical contracts: {len(target):,}",
    f"- Settlement dates: {target['event_date'].nunique():,}",
    f"- No-look-ahead market rows: {len(market):,}",
    f"- Admissible deterministic weather rows: {len(weather):,}",
    f"- March–May exact keys checked against 19b: {len(old_inter):,}",
    f"- June exact keys checked against 18r: {len(june_inter):,}",
    "",
    (
        "This adapter stage resolves legacy schemas, reconstructs "
        "canonical event boundaries, validates timestamps and removes "
        "archived Gaussian-bridge fields before aggregation."
    ),
]

report_path = REPORT / "18sA_canonical_source_adapters_report.md"
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows = []
for root in [OUT, REPORT]:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18sA_sha256_manifest.csv":
            continue
        manifest_rows.append(
            {
                "path": str(path.relative_to(ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha(path),
            }
        )

manifest_path = OUT / "18sA_sha256_manifest.csv"
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(json.dumps(summary, indent=2))
print("18sA canonical source adapter stage: PASS")

18k target: data/processed/18k_full_hko_contract_event_target_panel.csv (803 rows)
18o target: data/processed/18o_june_2026_hko_realised_outcomes/18o_june_2026_contract_outcomes.csv (330 rows)


18l market: data/processed/18l_full_hko_contract_event_market_scoring_panel.csv (2,921 rows)
18p market: data/processed/18p_june_2026_clob_market_price_recovery/18p_june_2026_market_scoring_panel.csv (1,265 rows)


19a weather: data/processed/19a_hko_ecmwf_contract_event_probability_panel.csv (3,212 rows)
18q weather: data/processed/18q_june_2026_ecmwf_single_run_forecasts/18q_june_2026_contract_event_membership.csv (1,320 rows)
19b common: data/processed/19b_common_support_market_vs_ecmwf_panel.csv (2,635 rows)
18r common: data/processed/18r_june_2026_market_weather_common_support/18r_june_2026_common_support_contract_panel.csv (1,254 rows)
march_may_baseline: canonical bounds ready; reconstructed rows=146


june_external_extension: canonical bounds ready; reconstructed rows=0
self-test-1 identifier resolved with market_id column: market_id_market
self-test-2 identifier resolved with condition_id column: condition_id_x
self-test-3 identifier resolved with selected_yes_token_id column: yes_token_id_market
self-test-4 identifier resolved with market_slug column: market_slug_x
mapped_common_keyset four-route self-test: PASS
19b common support identifier resolved with selected_yes_token_id column: selected_yes_token_id
18r common support identifier resolved with market_id column: market_id


{
  "step": "18sA",
  "generated_at_utc": "2026-07-21T20:55:17.894533+00:00",
  "verdict": "PASS",
  "canonical_target_rows": 1133,
  "canonical_target_dates": 103,
  "canonical_market_rows": 4186,
  "canonical_weather_rows": 4125,
  "march_may_common_keys": 2635,
  "june_common_keys": 1254,
  "march_may_common_keys_equal_19b": true,
  "june_common_keys_equal_18r": true,
  "probability_bridge_retained": false,
  "final_modelling_split_assigned": false,
  "outcome_availability_history_applied": false,
  "issue_rows": 0
}
18sA canonical source adapter stage: PASS
